# Dose Calculations

This section will cover the calculations to understand how much radiation is actually emitted from the device and its potential risk. As determined from the Katz-Penfold calculations, [katz-penfold.ipynb], the actual beta particles are quickly stopped by the water (max of ~1.1 cm distance). This means the only potential risk is in the Bremsstrahlung radiation emitted from these charged particles. To determine the safety of this project, the dose produced by these rays must be calculated. The first step of this process is to determine the total beta power emitted by the radioactive source.

In [ ]:
#code to compute total beta power
activity = 3700
sr_mean_energy = 0.196 #MeV
y_mean_energy = 0.933 #MeV

def compute_beta_power(activity, sr_mean_energy, y_mean_energy):
    # Calculate the total beta power
    total_beta_power = activity * (sr_mean_energy + y_mean_energy)
    return total_beta_power * 1.60218e-13  # Convert MeV/s to Joules/s (watts)
print("Total beta power (W):", compute_beta_power(activity, sr_mean_energy, y_mean_energy))

Total beta power (W): 6.692786514000001e-10


Now, the next step of this process is to determine the strength of the Bremsstrahlung radiation. This will be done via the Bremsstrahlung fraction:
$\frac{radiative}{total stopping power}$

As determined from ESTAR, the endpoint energy (2.28 MeV, worst-case), has a RLOSS (radiative) value of 0.0312 MeV*$cm^2$/g, and a TLOSS (total stopping power) value of 1.860 MeV*$cm^2$/g. This fraction will result in the amount of beta power that is actually emitted by the rays.


In [6]:
#code to compute bremsstrahlung fraction and beta power amount
def compute_bremsstrahlung_fraction(R, T):
    return R/T
def brem_beta_power(activity, sr_mean_energy, y_mean_energy, R, T):
    return compute_beta_power(activity, sr_mean_energy, y_mean_energy) * compute_bremsstrahlung_fraction(R, T)
RLOSS = 0.0312
TLOSS = 1.86
print(str(brem_beta_power(activity, sr_mean_energy, y_mean_energy, RLOSS, TLOSS)) + " W")

1.1226609636387097e-11 W


Now, the bremmstrahlung power has been determined to be 1.12E-11~ watts. The next step is to convert this value to a dose rate and compare it with legal limits. The definition of dose is energy absorbed per mass. Thus, to determine the dose rate, the power (1.122E-11 Watts) must be divided by the mass absorbing it. To be ultra-conservative, the mass that will absorb all of the radiation (in a real scenario, this is extrememly unlikely, as the rays radiate in all directions), will be ~1 kg of tissue. 

In [1]:
#code to compute dose rate from watts and mass
def compute_dose_rate(watts, mass):
    return watts / mass  # Dose rate in Gy/s (J/kg/s)
def convert_units_to_mrem(gy_per_s):
    mrem = gy_per_s * 3600 #(gy/s to Gy/h)
    mrem = mrem * 100 #(Gy/h to rem/h)
    mrem = mrem * 1000 #(rem/h to mrem/h)
    return mrem
mrem_hr = convert_units_to_mrem(compute_dose_rate(brem_beta_power(activity, sr_mean_energy, y_mean_energy, RLOSS, TLOSS), 1))
print("Dose rate (mrem/h):", mrem_hr)
print("The mrem/h produced is " + str(0.5/mrem_hr) + " times less than the limit of .5 mrem/h.")
hours_per_year = 500   # conservative estimate of how much I will be working under the source in a year
annual = mrem_hr * hours_per_year
print("Annual dose (mrem/yr):", annual)
print("Times under 100 mrem/yr limit:", 100/annual)

NameError: name 'brem_beta_power' is not defined

As seen by the final results, the radiation is safely under any rules and limitations. ISEF limits radiation exposure to to 0.5 mrem per hour at most, and 100 mrem per year. The amount produced here is 123.7 times lower and 49.5 times lower respectively. Furthermore, these calculations are deliberately conservative, (1kg of tissue, maximum possible energy, and overestimating operating hours), the true amount would be significantly lower.